# PongAI — Demo Bundle Export

Produces everything the frontend needs for one demo match, as a **self-consistent, frame-synced set**.

| output | contents |
|---|---|
| `{id}_main.mp4` | the match, web-encoded, no overlay |
| `{id}_left.mp4` | cropped to the left player, COCO skeleton overlaid |
| `{id}_right.mp4` | cropped to the right player, COCO skeleton overlaid |
| `{id}.json` | shots + kinematics + metadata, timestamps aligned to the videos |

### The constraint everything else follows from

The three videos play **side by side, synced by `currentTime`**. That only works if all three have **identical frame count, fps and duration**.

So every frame in range is rendered — including the ~43% of a match with no pose data. Those frames show the crop with no skeleton rather than being skipped. Skipping them would silently shift every timestamp after the first gap.

Output fps matches the source. `game_1` is 120fps, so the outputs are 120fps.

### Size

Full `game_1` is 12.3 min. At 360×540 / CRF 28 that is roughly **130 MB per player video, ~260 MB for the pair**, plus the main. Workable but heavy for a web demo.

`CLIP_START_S` / `CLIP_END_S` cut a window instead — and when you clip, the shot data is filtered and **rebased** to match, so the bundle stays consistent. A 3-minute clip lands near 35 MB per player.

No GPU needed. This reads cached pose and renders; it runs no models.


## 1 · Mount

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2 · Config

In [2]:
BASE     = "/content/drive/MyDrive/tt_coach"
VIDEO_ID = "game_1"

# --- clip window -------------------------------------------------------------
# None, None  = the whole match (game_1: 12.3 min, ~260 MB for the two players)
# A 3-minute window lands near 35 MB per player and loads far faster on the web.
CLIP_START_S = None
CLIP_END_S   = None

# --- crop / render -----------------------------------------------------------
OUT_W, OUT_H = 360, 540      # 2:3 portrait — player boxes are tall
CROP_ZOOM    = 1.9           # crop height = median box height x this
EMA_ALPHA    = 0.12          # crop-centre smoothing; lower = smoother, laggier
CRF          = 28            # 26 sharper/larger, 30 smaller/softer
KP_THRESH    = 0.30          # hide keypoints below this confidence
DRAW_TRAIL   = True          # fading wrist trail, ~1s
TRAIL_FRAMES = 120

import json, subprocess, shutil, time
from pathlib import Path
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm

BASE   = Path(BASE)
META   = BASE/"derived/meta"
STREAM = BASE/"derived/pose_stream"
ANALYSED = BASE/"derived/analysed"/VIDEO_ID
SRC    = BASE/f"raw/videos/{VIDEO_ID}.mp4"
OUTDIR = BASE/"outputs/demo"/VIDEO_ID
OUTDIR.mkdir(parents=True, exist_ok=True)
TMP    = Path("/content/_demo"); TMP.mkdir(exist_ok=True)

# COCO-17
NOSE=0; L_SHO,R_SHO,L_ELB,R_ELB,L_WRI,R_WRI = 5,6,7,8,9,10
L_HIP,R_HIP,L_KNE,R_KNE,L_ANK,R_ANK = 11,12,13,14,15,16
EDGES = [(5,6),(5,7),(7,9),(6,8),(8,10),(5,11),(6,12),(11,12),
         (11,13),(13,15),(12,14),(14,16),(0,5),(0,6)]

BRAND      = (2, 95, 245)      # #f55f02 in BGR
BRAND_DIM  = (60,120,200)
WRIST_COL  = (60, 60, 255)

for p in (SRC, STREAM/f"{VIDEO_ID}.npz", ANALYSED/"shots.parquet"):
    assert p.exists(), f"missing: {p}"

cap = cv2.VideoCapture(str(SRC))
SRC_FPS = cap.get(cv2.CAP_PROP_FPS)
SRC_N   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
SRC_W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
SRC_H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

F0 = 0 if CLIP_START_S is None else int(CLIP_START_S*SRC_FPS)
F1 = SRC_N if CLIP_END_S is None else min(SRC_N, int(CLIP_END_S*SRC_FPS))
N_OUT = F1 - F0

print(f"source : {SRC_W}x{SRC_H} @ {SRC_FPS:.0f}fps, {SRC_N:,} frames "
      f"({SRC_N/SRC_FPS/60:.1f} min)")
print(f"render : frames {F0:,}-{F1:,} = {N_OUT:,} frames "
      f"({N_OUT/SRC_FPS/60:.1f} min) @ {SRC_FPS:.0f}fps")
print(f"output : {OUT_W}x{OUT_H} per player")
est = N_OUT/SRC_FPS * 1.4/8
print(f"est size: ~{est:.0f} MB per player video")

source : 1920x1080 @ 120fps, 88,599 frames (12.3 min)
render : frames 0-88,599 = 88,599 frames (12.3 min) @ 120fps
output : 360x540 per player
est size: ~129 MB per player video


## 3 · Load pose and shots

`pose_stream` covers only the active regions the pipeline extracted. Frames outside them have no pose — handled in the next cell rather than skipped.

In [3]:
d = np.load(STREAM/f"{VIDEO_ID}.npz", allow_pickle=True)
FIDX = d["frame_idx"].astype(int)
KP   = d["keypoints"].astype(np.float32)   # (n, 2, 17, 2) VIDEO PIXELS
SC   = d["scores"].astype(np.float32)      # (n, 2, 17)
BX   = d["boxes"].astype(np.float32)       # (n, 2, 4)
DT   = d["detected"]                       # (n, 2)

shots = pd.read_parquet(ANALYSED/"shots.parquet").sort_values("frame")

# frame -> index into the pose arrays, -1 where absent
LUT = np.full(SRC_N, -1, np.int32)
LUT[FIDX] = np.arange(len(FIDX))

cov = (LUT[F0:F1] >= 0).mean()
print(f"pose covers {len(FIDX):,} frames "
      f"({cov:.0%} of the render range)")
print(f"shots in range: {((shots.frame>=F0)&(shots.frame<F1)).sum()} "
      f"of {len(shots)}")
print(f"\nframes with no pose render as the crop with NO skeleton — "
      f"they are NOT skipped,\nbecause dropping them would shift every "
      f"later timestamp out of sync.")

pose covers 20,144 frames (23% of the render range)
shots in range: 164 of 164

frames with no pose render as the crop with NO skeleton — they are NOT skipped,
because dropping them would shift every later timestamp out of sync.


## 4 · Crop geometry

Three details decide whether this is watchable.

**Fixed zoom.** The detector box grows and shrinks as a player moves toward and away from the camera. Following it makes the player constantly rescale on screen, which is distracting and makes posture impossible to compare between shots. One crop size is computed for the whole match from the median box height; only the *centre* follows the player.

**Smoothed centre.** Raw per-frame boxes jitter several pixels. Magnified into a zoomed crop that is a violent shake. An EMA at α≈0.12 removes it.

**Gap filling.** Where there is no detection, the centre is carried forward so the crop holds still rather than snapping to a default.

In [4]:
def build_crop_track(player_idx):
    """-> (cx, cy) per source frame, smoothed; plus the fixed crop size."""
    cx = np.full(SRC_N, np.nan, np.float64)
    cy = np.full(SRC_N, np.nan, np.float64)
    heights = []
    for i, f in enumerate(FIDX):
        if not DT[i, player_idx]:
            continue
        x1, y1, x2, y2 = BX[i, player_idx]
        if x2 <= x1 or y2 <= y1:
            continue
        cx[f] = (x1+x2)/2; cy[f] = (y1+y2)/2
        heights.append(y2-y1)

    if not heights:
        raise RuntimeError(f"no detections for player {player_idx}")

    crop_h = float(np.median(heights)) * CROP_ZOOM
    crop_w = crop_h * (OUT_W/OUT_H)
    crop_h = min(crop_h, SRC_H); crop_w = min(crop_w, SRC_W)

    # fill gaps: forward then backward, so the crop holds instead of jumping
    idx = np.arange(SRC_N)
    good = ~np.isnan(cx)
    cx = np.interp(idx, idx[good], cx[good])
    cy = np.interp(idx, idx[good], cy[good])

    # EMA smoothing
    sx = np.empty_like(cx); sy = np.empty_like(cy)
    sx[0], sy[0] = cx[0], cy[0]
    for k in range(1, SRC_N):
        sx[k] = sx[k-1] + EMA_ALPHA*(cx[k]-sx[k-1])
        sy[k] = sy[k-1] + EMA_ALPHA*(cy[k]-sy[k-1])

    # clamp so the crop rect stays inside the frame
    sx = np.clip(sx, crop_w/2, SRC_W-crop_w/2)
    sy = np.clip(sy, crop_h/2, SRC_H-crop_h/2)
    return sx, sy, crop_w, crop_h


TRACK = {}
for pi, side in ((0,"left"), (1,"right")):
    sx, sy, cw, ch = build_crop_track(pi)
    TRACK[side] = dict(cx=sx, cy=sy, w=cw, h=ch, pi=pi)
    print(f"{side:<6} crop {cw:.0f}x{ch:.0f} px  "
          f"({cw/SRC_W:.0%} x {ch/SRC_H:.0%} of frame)  "
          f"scale {OUT_W/cw:.2f}x")

left   crop 740x1080 px  (39% x 100% of frame)  scale 0.49x
right  crop 963x1080 px  (50% x 100% of frame)  scale 0.37x


## 5 · Render

One decode pass writes all three outputs. Reading the source once instead of three times is most of the speed here.

In [5]:
def draw_skeleton(img, kp, sc, cx0, cy0, scale, alpha=1.0):
    """kp/sc in video pixels -> draw into the crop-local canvas."""
    def P(j):
        return (int((kp[j,0]-cx0)*scale), int((kp[j,1]-cy0)*scale))
    col = BRAND if alpha >= 1.0 else BRAND_DIM
    for a, b in EDGES:
        if sc[a] >= KP_THRESH and sc[b] >= KP_THRESH:
            pa, pb = P(a), P(b)
            cv2.line(img, pa, pb, (0,0,0), 4, cv2.LINE_AA)     # outline
            cv2.line(img, pa, pb, col,     2, cv2.LINE_AA)
    for j in range(17):
        if sc[j] < KP_THRESH: continue
        p = P(j)
        r = 6 if j in (L_WRI, R_WRI) else 3
        c = WRIST_COL if j in (L_WRI, R_WRI) else col
        cv2.circle(img, p, r+1, (0,0,0), -1, cv2.LINE_AA)
        cv2.circle(img, p, r,   c,     -1, cv2.LINE_AA)
    return img


writers, paths = {}, {}
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
paths["main"]  = TMP/f"{VIDEO_ID}_main_raw.mp4"
paths["left"]  = TMP/f"{VIDEO_ID}_left_raw.mp4"
paths["right"] = TMP/f"{VIDEO_ID}_right_raw.mp4"
MAIN_W, MAIN_H = 1280, int(1280*SRC_H/SRC_W)
writers["main"]  = cv2.VideoWriter(str(paths["main"]),  fourcc, SRC_FPS, (MAIN_W, MAIN_H))
writers["left"]  = cv2.VideoWriter(str(paths["left"]),  fourcc, SRC_FPS, (OUT_W, OUT_H))
writers["right"] = cv2.VideoWriter(str(paths["right"]), fourcc, SRC_FPS, (OUT_W, OUT_H))

trail = {"left": [], "right": []}
cap = cv2.VideoCapture(str(SRC))
cap.set(cv2.CAP_PROP_POS_FRAMES, F0)

t0 = time.time()
written = 0
for f in tqdm(range(F0, F1), desc="render"):
    ok, frame = cap.read()
    if not ok:
        print(f"\n  decode stopped at frame {f}"); break

    writers["main"].write(cv2.resize(frame, (MAIN_W, MAIN_H)))
    pi_idx = LUT[f]

    for side in ("left", "right"):
        t = TRACK[side]; pi = t["pi"]
        cx0 = t["cx"][f] - t["w"]/2
        cy0 = t["cy"][f] - t["h"]/2
        x0, y0 = int(round(cx0)), int(round(cy0))
        x1, y1 = x0 + int(round(t["w"])), y0 + int(round(t["h"]))
        x0c, y0c = max(0, x0), max(0, y0)
        x1c, y1c = min(SRC_W, x1), min(SRC_H, y1)

        crop = frame[y0c:y1c, x0c:x1c]
        if crop.size == 0:
            canvas = np.zeros((OUT_H, OUT_W, 3), np.uint8)
        else:
            canvas = cv2.resize(crop, (OUT_W, OUT_H), interpolation=cv2.INTER_AREA)

        scale = OUT_W / t["w"]
        has_pose = pi_idx >= 0 and DT[pi_idx, pi]

        if has_pose:
            kp, sc = KP[pi_idx, pi], SC[pi_idx, pi]
            if DRAW_TRAIL:
                w_j = R_WRI if sc[R_WRI] >= sc[L_WRI] else L_WRI
                if sc[w_j] >= KP_THRESH:
                    trail[side].append((kp[w_j,0], kp[w_j,1]))
                    if len(trail[side]) > TRAIL_FRAMES: trail[side].pop(0)
                tr = trail[side]
                for k in range(1, len(tr)):
                    a = k/len(tr)
                    pa = (int((tr[k-1][0]-x0c)*scale), int((tr[k-1][1]-y0c)*scale))
                    pb = (int((tr[k][0]  -x0c)*scale), int((tr[k][1]  -y0c)*scale))
                    cv2.line(canvas, pa, pb,
                             tuple(int(c*a) for c in WRIST_COL),
                             max(1, int(3*a)), cv2.LINE_AA)
            draw_skeleton(canvas, kp, sc, x0c, y0c, scale)
        else:
            trail[side].clear()
            canvas = cv2.convertScaleAbs(canvas, alpha=0.72, beta=0)   # dim

        writers[side].write(canvas)
    written += 1

cap.release()
for w in writers.values(): w.release()
el = time.time()-t0
print(f"\n{written:,} frames in {el/60:.1f} min "
      f"({written/max(el,1):.0f} fps processing)")
assert written == N_OUT, (
    f"FRAME COUNT MISMATCH: wrote {written}, expected {N_OUT}. "
    "The three videos would not stay in sync.")
print("frame counts match — the three videos will stay in sync.")

render:   0%|          | 0/88599 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 6 · Encode for the web

`mp4v` from OpenCV does not play in browsers. Re-encode to h264 with `faststart` so playback can begin before the whole file downloads.

In [ ]:
finals = {}
for key, raw in paths.items():
    out = OUTDIR/f"{VIDEO_ID}_{key}.mp4"
    crf = CRF if key != "main" else CRF + 1
    subprocess.run([
        "ffmpeg","-y","-loglevel","error","-i",str(raw),
        "-c:v","libx264","-preset","medium","-crf",str(crf),
        "-pix_fmt","yuv420p","-movflags","+faststart",
        "-r",f"{SRC_FPS:.0f}", str(out)], check=True)
    finals[key] = out
    raw.unlink(missing_ok=True)
    print(f"  {out.name:<24} {out.stat().st_size/1e6:6.1f} MB")

# verify all three agree on frames, fps and duration
print("\nsync check")
ref = None
for key, p in finals.items():
    cap = cv2.VideoCapture(str(p))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    print(f"  {key:<6} {n:>7,} frames  {fps:6.2f} fps  {n/fps:7.2f}s  {w}x{h}")
    if ref is None: ref = (n, round(fps,2))
    else: assert (n, round(fps,2)) == ref, f"{key} does not match the main video"
print("  all three match.")

## 7 · Shot data

Filtered to the render window and **rebased** so `timestamp_s` is relative to the start of the clip — otherwise the timeline would point past the end of a clipped video.

In [ ]:
s = shots[(shots.frame >= F0) & (shots.frame < F1)].copy().reset_index(drop=True)
s["frame"] = s["frame"] - F0
s["timestamp_s"] = s["frame"] / SRC_FPS

# renumber rallies from 0 and reindex shots within them
s["rally_id"] = pd.factorize(s["rally_id"])[0]
s["shot_index"] = s.groupby("rally_id").cumcount()
s["rally_length"] = s.groupby("rally_id")["rally_id"].transform("size")

KINEMATICS = ["backswing_amplitude","peak_wrist_speed","time_to_peak",
              "contact_height","elbow_angle","elbow_range","trunk_lean",
              "trunk_rotation","table_distance","stance_width","knee_angle",
              "follow_through","recovery_time"]
FIELDS = (["rally_id","shot_index","player","frame","timestamp_s","shot_class",
           "class_confidence","technique","abstain","detect_confidence",
           "rally_length","pose_confidence","detected"] + KINEMATICS)

recs = []
for r in s.itertuples():
    rec = {}
    for k in FIELDS:
        v = getattr(r, k, None)
        if isinstance(v, (np.integer,)):   v = int(v)
        elif isinstance(v, (np.floating,)):
            v = None if not np.isfinite(v) else round(float(v), 4)
        elif isinstance(v, (np.bool_,)):   v = bool(v)
        rec[k] = v
    recs.append(rec)

bundle = {
    "video_id": VIDEO_ID,
    "video_url": f"/demo/{VIDEO_ID}_main.mp4",
    "video_left_url":  f"/demo/{VIDEO_ID}_left.mp4",
    "video_right_url": f"/demo/{VIDEO_ID}_right.mp4",
    "duration_s": round(N_OUT/SRC_FPS, 3),
    "source_fps": round(SRC_FPS, 2),
    "width": MAIN_W, "height": MAIN_H,
    "clip": {"start_frame": F0, "end_frame": F1,
             "start_s": round(F0/SRC_FPS,3), "is_clipped": F0 != 0 or F1 != SRC_N},
    "shots": recs,
}
out_json = OUTDIR/f"{VIDEO_ID}.json"
out_json.write_text(json.dumps(bundle, indent=1))
print(f"{out_json.name}  {out_json.stat().st_size/1e6:.2f} MB, {len(recs)} shots")

mix = s.shot_class.value_counts()
print(f"\nrallies {s.rally_id.nunique()}, mean {len(s)/max(s.rally_id.nunique(),1):.1f} shots")
for c in ["serve","attack","control","defence"]:
    print(f"  {c:<9} {int(mix.get(c,0)):>4}")
print(f"suppressed (abstain) {int(s.abstain.sum())} of {len(s)}")

# invariants that would break the frontend if violated
assert s.timestamp_s.is_monotonic_increasing, "timestamps not monotonic"
assert s.timestamp_s.max() <= N_OUT/SRC_FPS + 0.01, "a shot lands past the video end"
assert (s.groupby("rally_id").shot_index.first() == 0).all(), "rally shot_index does not start at 0"
print("\ninvariants pass.")

## 8 · Copy to the frontend

In [ ]:
print("Copy these four files into the Next.js app at public/demo/ :\n")
tot = 0
for p in [OUTDIR/f"{VIDEO_ID}.json", finals["main"], finals["left"], finals["right"]]:
    mb = p.stat().st_size/1e6; tot += mb
    print(f"  {p}   ({mb:.1f} MB)")
print(f"\n  total {tot:.0f} MB")
if tot > 100:
    print(f"""
  Over 100 MB. Options:
    - set CLIP_START_S / CLIP_END_S and re-run — the shot data is rebased
      automatically, so the bundle stays consistent
    - raise CRF to 30
    - drop OUT_W/OUT_H to 300x450
    - host the videos on R2/S3 and point video_url at them rather than
      committing them to the repo""")

print(f"""
Frontend notes
  - three <video> elements, all {SRC_FPS:.0f}fps, identical frame count
  - drive left/right from the main video's currentTime; do not let them
    free-run or they will drift
  - the left/right videos already carry the skeleton, so no canvas overlay
    and no pose_track sidecar is needed for this demo
  - timestamps in the JSON are relative to the start of the clip""")

---
## What this produces

A complete, self-consistent demo for one match. The three videos share a frame count and fps, so the frontend syncs them on `currentTime` alone.

**To clip:** set `CLIP_START_S` / `CLIP_END_S` and re-run. Shots are filtered, timestamps rebased, and rallies renumbered — the bundle cannot drift out of agreement with the video.

**Runtime:** roughly 25–60 minutes for the full 12.3-minute match, dominated by decoding 88,599 frames at 1080p. Clipping cuts it proportionally.

**Note the trade-off:** baking the skeleton into the video is simpler than shipping pose data and drawing in the browser, but it fixes the overlay — no layer toggles, no isolated-skeleton view. Fine for a demo; revisit if the interactive version matters later.
